# Notebook 04 — Rescue-Aware Decision Engine for Predicted Late-Stage Clone Selection

## Goal

Convert predicted late-stage clone performance from Notebook03 into practical CLD selection decisions.

This notebook implements:

- pass / rescue / fail bucket assignment
- biosimilar and novel/ADC decision modes
- Stage 2 re-ranking
- final Top-10 / Top-3 clone selection
- rescue-aware decision validation

A rescue clone is not a final winner yet.  
It is a clone with enough upside to justify further process optimization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

scenario = "legacy"   # or "optimized"
n_clones = 5000

TOP_N = 10
TOP_STAGE2 = 3

print("Scenario:", scenario)
print("n_clones:", n_clones)
print("Top-N final:", TOP_N)
print("Top-N stage2:", TOP_STAGE2)

Scenario: legacy
n_clones: 5000
Top-N final: 10
Top-N stage2: 3


## Step 1 — Load prediction table from Notebook 03b

Here we load the prediction table produced in Notebook 03b.

This table should already contain:

- predicted late qP
- predicted qP drop
- predicted late aggregation
- predicted stable probability
- true late outcomes for offline simulation only

In [2]:
PRED_PATH = Path("../data/synthetic/processed") / f"predictions_03b_qp_{n_clones}_{scenario}.csv"
print("Loading predictions:", PRED_PATH)

df = pd.read_csv(PRED_PATH)
print("Shape:", df.shape)
display(df.head())

Loading predictions: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 19)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.498165,2.673129e-08,8.435996,0.238692,0,0.206868,3.215605e-08,4.404966,1,0.001494,0.003219,0.000312,0.506117,0.981713,0.996769,0.710487,0.568590,0
1,CLONE_2587,0.373417,8.407121e-08,3.130765,0.464397,0,0.600740,5.393417e-08,3.662670,0,0.007656,0.000000,0.013124,0.921942,0.000000,1.000000,0.435364,0.416664,0
2,CLONE_2654,0.461606,2.770586e-08,7.582349,0.256485,0,0.156409,3.459109e-08,10.144181,1,0.001523,0.004162,0.000529,0.627981,0.737814,0.995823,0.688798,0.538817,0
3,CLONE_1056,0.348761,7.746555e-08,6.885601,0.507813,1,0.465880,6.392818e-08,2.383546,0,0.012355,0.000000,0.011648,0.991742,0.538743,1.000000,0.382442,0.594558,0
4,CLONE_0706,0.644735,1.744327e-07,8.275030,0.018172,0,0.678633,1.526521e-07,5.032915,0,0.004539,0.039329,0.033316,0.017551,0.935723,0.960529,0.979289,0.407241,0


## Step 2 — Check schema compatibility

Before applying selection rules, we verify that the prediction table matches the expected 03b output schema.

This prevents silent notebook mismatch caused by:
- outdated file names
- old column naming
- partial notebook updates

In [3]:
required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_stable_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)
assert len(missing) == 0, f"04b is missing required columns: {missing}"

optional_cols = [
    "pred_super_prob",
    "pred_aggr_prob",
    "pred_rescue_score",
    "pred_rescue_label",
]

present_optional = [c for c in optional_cols if c in df.columns]
print("Optional columns found:", present_optional)

if "pred_rescue_score" not in df.columns:
    print("[WARN] pred_rescue_score missing. Setting to 0.0.")
    df["pred_rescue_score"] = 0.0

if "pred_rescue_label" not in df.columns:
    print("[WARN] pred_rescue_label missing. Setting to 0.")
    df["pred_rescue_label"] = 0

Missing required columns: []
Optional columns found: ['pred_super_prob', 'pred_aggr_prob', 'pred_rescue_score', 'pred_rescue_label']


## Step 3 — Define helper functions

We use helper functions for:

- z-score normalization
- ranking utility calculation
- top-k overlap
- elite capture metrics

These help compare decision quality across selection modes.

In [4]:
def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def make_pred_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["pred_late_qp"])
        - w_drop * z(df["pred_qp_drop"])
        - w_agg * z(df["pred_late_agg"])
    )

def make_true_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["true_late_qp"])
        - w_drop * z(df["true_qp_drop"])
        - w_agg * z(df["true_late_agg"])
    )

def topk_overlap(true_scores, pred_scores, k):
    true_top = set(pd.Series(true_scores).nlargest(k).index)
    pred_top = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top & pred_top) / k

def precision_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    return float(top[elite_col].mean())

def ndcg_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    rel = top[elite_col].values.astype(float)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))
    dcg = float(np.sum(rel * discounts))
    ideal = np.sort(rel)[::-1]
    idcg = float(np.sum(ideal * discounts)) + 1e-12
    return dcg / idcg

In [5]:
def summarize_bucket(df, bucket_col):
    return df.groupby(bucket_col)[[
        "pred_late_qp",
        "pred_qp_drop",
        "pred_late_agg",
        "pred_stable_prob",
        "pred_rescue_score",
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_stable_label",
    ]].mean()

def summarize_final_selection(name, top10, top3, bucket_col):
    return {
        "mode": name,
        "top10_n": len(top10),
        "top3_n": len(top3),
        "top10_rescue_count": int((top10[bucket_col] == "rescue").sum()),
        "top3_rescue_count": int((top3[bucket_col] == "rescue").sum()),
        "top10_true_stable_rate": float(top10["true_stable_label"].mean()),
        "top3_true_stable_rate": float(top3["true_stable_label"].mean()),
        "top10_mean_true_late_qp": float(top10["true_late_qp"].mean()),
        "top10_mean_true_qp_drop": float(top10["true_qp_drop"].mean()),
        "top10_mean_true_late_agg": float(top10["true_late_agg"].mean()),
        "top10_mean_pred_rescue_score": float(top10["pred_rescue_score"].mean()),
    }

def apply_rescue_guardrail(
    stage2_df,
    bucket_col,
    score_col,
    top_n=10,
    top_stage2=3,
    min_rescue_top10=1,
):
    """
    Keep Top3 as pure ranking.
    For Top10, use pure ranking by default.
    If no rescue clone appears in Top10, replace the lowest-ranked non-Top3 pass clone
    with the best rescue candidate.
    """
    ranked = stage2_df.sort_values(score_col, ascending=False).copy()

    # Top3 is always pure ranking
    top3 = ranked.head(top_stage2).copy()

    # Top10 starts as pure ranking
    top10 = ranked.head(top_n).copy()

    if min_rescue_top10 <= 0:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    current_rescue_n = int((top10[bucket_col] == "rescue").sum())

    if current_rescue_n >= min_rescue_top10:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    protected_ids = set(top3["clone_id"])
    top10_ids = set(top10["clone_id"])

    rescue_pool = stage2_df[
        (stage2_df[bucket_col] == "rescue") &
        (~stage2_df["clone_id"].isin(top10_ids)) &
        (~stage2_df["clone_id"].isin(protected_ids))
    ].copy()

    if len(rescue_pool) == 0:
        return top10.reset_index(drop=True), top3.reset_index(drop=True)

    need_n = min_rescue_top10 - current_rescue_n

    rescue_add = rescue_pool.sort_values(
        ["pred_rescue_score", score_col],
        ascending=[False, False]
    ).head(need_n)

    replace_candidates = top10[
        (~top10["clone_id"].isin(protected_ids)) &
        (top10[bucket_col] != "rescue")
    ].sort_values(score_col, ascending=True).head(len(rescue_add))

    top10_new = pd.concat(
        [
            top10[~top10["clone_id"].isin(replace_candidates["clone_id"])],
            rescue_add,
        ],
        ignore_index=True,
    ).sort_values(score_col, ascending=False).head(top_n)

    return top10_new.reset_index(drop=True), top3.reset_index(drop=True)

## Step 4 — Define offline evaluation utilities

Because this is a synthetic dataset, we can define an offline “true utility” score
to evaluate how good the selected clones really are.

This is **evaluation only** and would not be available in real deployment.

In [6]:
# biosimilar-style evaluation utility
W_BIO = dict(w_qp=1.0, w_drop=1.0, w_agg=0.2)

# novel / ADC-style evaluation utility
W_NOVEL = dict(w_qp=1.0, w_drop=1.0, w_agg=1.0)

df["true_util_bio"] = make_true_utility(df, **W_BIO)
df["true_util_novel"] = make_true_utility(df, **W_NOVEL)

df["pred_util_bio"] = make_pred_utility(df, **W_BIO)
df["pred_util_novel"] = make_pred_utility(df, **W_NOVEL)

display(df[[
    "clone_id",
    "true_util_bio", "pred_util_bio",
    "true_util_novel", "pred_util_novel"
]].head())

,clone_id,true_util_bio,pred_util_bio,true_util_novel,pred_util_novel
0,CLONE_1502,1.550478,-0.977650,2.009245,-1.755465
1,CLONE_2587,-0.804406,0.773624,-0.161434,1.859755
2,CLONE_2654,1.502466,-0.568919,0.537017,-1.046813
3,CLONE_1056,0.098570,0.649212,1.058963,0.416115
4,CLONE_0706,-1.355847,-1.890491,-1.052909,-2.611753


## Step 5A — Decision mode: Biosimilar

In biosimilar mode, the selection logic is:

1. remove clones with poor predicted quality
2. remove clones with low predicted stability probability
3. rank the remaining clones by predicted late qP

This reflects a setting where:
- productivity matters strongly
- stability matters strongly
- aggregation still matters, but slightly less than in novel mode

In [7]:
# ==================================================
# Biosimilar mode — rescue-aware 3-bucket engine
# ==================================================

BIO_AGG_PASS_MAX = 15.0
BIO_STABLE_PASS_MIN = 0.50

BIO_AGG_RESCUE_MAX = 18.0
BIO_STABLE_RESCUE_MIN = 0.25
BIO_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.70)

BIO_STAGE2_PASS_N = 25
BIO_STAGE2_RESCUE_N = 5

bio = df.copy()

bio["bio_pass"] = (
    (bio["pred_late_agg"] <= BIO_AGG_PASS_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_PASS_MIN)
)

bio["bio_rescue"] = (
    (~bio["bio_pass"]) &
    (bio["pred_rescue_score"] >= BIO_RESCUE_SCORE_MIN) &
    (bio["pred_late_agg"] <= BIO_AGG_RESCUE_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_RESCUE_MIN)
)

bio["bio_bucket"] = "fail"
bio.loc[bio["bio_rescue"], "bio_bucket"] = "rescue"
bio.loc[bio["bio_pass"], "bio_bucket"] = "pass"

bio_stage1_pass = bio[bio["bio_bucket"] == "pass"].copy()
bio_stage1_rescue = bio[bio["bio_bucket"] == "rescue"].copy()
bio_stage1_fail = bio[bio["bio_bucket"] == "fail"].copy()

print("=== Biosimilar Stage 1 ===")
print(bio["bio_bucket"].value_counts())

print("\n=== Biosimilar bucket diagnostics ===")
display(summarize_bucket(bio, "bio_bucket"))

bio_stage2_pass = bio_stage1_pass.sort_values(
    ["pred_late_qp", "pred_stable_prob"],
    ascending=[False, False]
).head(BIO_STAGE2_PASS_N).copy()

bio_stage2_rescue = bio_stage1_rescue.sort_values(
    ["pred_rescue_score", "pred_late_qp"],
    ascending=[False, False]
).head(BIO_STAGE2_RESCUE_N).copy()

bio_stage2 = pd.concat([bio_stage2_pass, bio_stage2_rescue], ignore_index=True)

bio_stage2["bio_final_score"] = (
    z(bio_stage2["pred_late_qp"])
    - 0.5 * z(bio_stage2["pred_qp_drop"])
    - 0.2 * z(bio_stage2["pred_late_agg"])
    + 0.7 * z(bio_stage2["pred_rescue_score"])
)

BIO_MIN_RESCUE_TOP10 = 1

bio_top10, bio_top3 = apply_rescue_guardrail(
    stage2_df=bio_stage2,
    bucket_col="bio_bucket",
    score_col="bio_final_score",
    top_n=TOP_N,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=BIO_MIN_RESCUE_TOP10,
)

print("\n=== Biosimilar Stage 2 / Final ===")
print("Stage2 pass:", len(bio_stage2_pass))
print("Stage2 rescue:", len(bio_stage2_rescue))
print("Final top10:", len(bio_top10))
print("Final top3:", len(bio_top3))

display(bio_top10[[
    "clone_id",
    "bio_bucket",
    "bio_final_score",
    "pred_rescue_score",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in bio_top10.columns]])

=== Biosimilar Stage 1 ===
bio_bucket
fail      758
rescue    160
pass       82
Name: count, dtype: int64

=== Biosimilar bucket diagnostics ===


,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_stable_label
bio_bucket,,,,,,,,,
fail,1.567428e-07,0.477066,5.778478,0.201773,0.404422,6.351721e-07,0.477744,5.722433,0.130607
pass,2.935525e-07,0.328114,6.297133,0.574652,0.508005,4.325392e-07,0.310092,6.834508,0.451220
rescue,9.320008e-08,0.390032,8.285623,0.355348,0.618298,9.690855e-08,0.387139,8.472761,0.200000



=== Biosimilar Stage 2 / Final ===
Stage2 pass: 25
Stage2 rescue: 5
Final top10: 10
Final top3: 3


,clone_id,bio_bucket,bio_final_score,pred_rescue_score,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_3254,pass,5.178587,0.787107,4.328425e-06,0.246238,8.356089,0.821561,7.103148e-06,0.089331,11.556387,1,0.864245,0.007259
1,CLONE_4625,pass,5.009523,1.000000,4.500575e-06,0.322967,8.670837,0.560659,1.238325e-05,0.084353,10.114743,1,0.756864,0.015903
2,CLONE_1619,pass,2.569500,0.358295,2.630590e-06,0.236190,3.356785,0.731156,2.544839e-06,0.164530,5.259631,1,0.854835,0.002945
3,CLONE_2759,pass,0.274401,0.481011,8.872743e-07,0.273353,10.806613,0.685999,1.094417e-06,0.220844,16.093701,1,0.647280,0.004383
4,CLONE_2689,pass,-0.033607,0.618017,2.165995e-07,0.302717,8.513324,0.701941,1.648067e-07,0.484972,13.559799,0,0.017261,0.000000
5,CLONE_3144,pass,-0.100794,0.649180,2.741725e-07,0.320466,8.292429,0.675962,2.349968e-07,0.419978,8.431879,0,0.325483,0.000000
6,CLONE_4335,pass,-0.204466,0.717382,3.366004e-07,0.345661,8.935721,0.527962,3.337148e-07,0.372991,10.260128,0,0.157209,0.001358
7,CLONE_0550,pass,-0.353280,0.495934,2.528852e-07,0.303989,6.943930,0.685301,2.055263e-07,0.440764,5.093851,0,0.242958,0.001302
8,CLONE_3581,pass,-0.379346,0.251555,3.571934e-07,0.268148,3.505295,0.785921,3.901131e-07,0.252349,5.063234,1,0.419366,0.000000
9,CLONE_1942,rescue,-0.450262,0.733136,8.271182e-08,0.353795,8.453091,0.379410,1.027260e-07,0.239874,8.372210,1,0.003232,0.000000


## Step 6A — Novel / ADC 3-bucket decision engine

In novel / ADC mode, quality is emphasized more strongly than in biosimilar mode.

So compared with biosimilar mode:

- aggregation threshold is stricter
- stability threshold is slightly stricter
- rescue margin is narrower
- productivity still matters, but low-quality clones are filtered more aggressively

We use the same 3-bucket logic:

- **pass** = strong candidates that clearly meet the criteria
- **rescue** = borderline but potentially valuable clones
- **fail** = clones that should be dropped

In [8]:
# ==================================================
# Novel / ADC mode — rescue-aware 3-bucket engine
# ==================================================

NOVEL_AGG_PASS_MAX = 10.0
NOVEL_STABLE_PASS_MIN = 0.55

NOVEL_AGG_RESCUE_MAX = 13.0
NOVEL_STABLE_RESCUE_MIN = 0.30
NOVEL_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.75)

NOVEL_STAGE2_PASS_N = 25
NOVEL_STAGE2_RESCUE_N = 5

novel = df.copy()

novel["novel_pass"] = (
    (novel["pred_late_agg"] <= NOVEL_AGG_PASS_MAX) &
    (novel["pred_stable_prob"] >= NOVEL_STABLE_PASS_MIN)
)

novel["novel_rescue"] = (
    (~novel["novel_pass"]) &
    (novel["pred_rescue_score"] >= NOVEL_RESCUE_SCORE_MIN) &
    (novel["pred_late_agg"] <= NOVEL_AGG_RESCUE_MAX) &
    (novel["pred_stable_prob"] >= NOVEL_STABLE_RESCUE_MIN)
)

novel["novel_bucket"] = "fail"
novel.loc[novel["novel_rescue"], "novel_bucket"] = "rescue"
novel.loc[novel["novel_pass"], "novel_bucket"] = "pass"

novel_stage1_pass = novel[novel["novel_bucket"] == "pass"].copy()
novel_stage1_rescue = novel[novel["novel_bucket"] == "rescue"].copy()
novel_stage1_fail = novel[novel["novel_bucket"] == "fail"].copy()

print("=== Novel / ADC Stage 1 ===")
print(novel["novel_bucket"].value_counts())

print("\n=== Novel / ADC bucket diagnostics ===")
display(summarize_bucket(novel, "novel_bucket"))

novel_stage2_pass = novel_stage1_pass.sort_values(
    ["pred_late_agg", "pred_late_qp", "pred_stable_prob"],
    ascending=[True, False, False]
).head(NOVEL_STAGE2_PASS_N).copy()

novel_stage2_rescue = novel_stage1_rescue.sort_values(
    ["pred_rescue_score", "pred_late_agg", "pred_late_qp"],
    ascending=[False, True, False]
).head(NOVEL_STAGE2_RESCUE_N).copy()

novel_stage2 = pd.concat([novel_stage2_pass, novel_stage2_rescue], ignore_index=True)

novel_stage2["novel_final_score"] = (
    z(novel_stage2["pred_late_qp"])
    - 0.6 * z(novel_stage2["pred_qp_drop"])
    - 1.0 * z(novel_stage2["pred_late_agg"])
    + 0.5 * z(novel_stage2["pred_rescue_score"])
)

NOVEL_MIN_RESCUE_TOP10 = 0

novel_top10, novel_top3 = apply_rescue_guardrail(
    stage2_df=novel_stage2,
    bucket_col="novel_bucket",
    score_col="novel_final_score",
    top_n=TOP_N,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=NOVEL_MIN_RESCUE_TOP10,
)

print("\n=== Novel / ADC Stage 2 / Final ===")
print("Stage2 pass:", len(novel_stage2_pass))
print("Stage2 rescue:", len(novel_stage2_rescue))
print("Final top10:", len(novel_top10))
print("Final top3:", len(novel_top3))

display(novel_top10[[
    "clone_id",
    "novel_bucket",
    "novel_final_score",
    "pred_rescue_score",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_top10.columns]])

=== Novel / ADC Stage 1 ===
novel_bucket
fail      839
rescue    123
pass       38
Name: count, dtype: int64

=== Novel / ADC bucket diagnostics ===


,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_stable_label
novel_bucket,,,,,,,,,
fail,1.525910e-07,0.467804,5.907451,0.219340,0.417328,5.855390e-07,0.466947,5.881248,0.140644
pass,4.757351e-07,0.314835,6.458077,0.629399,0.509515,7.683681e-07,0.286164,6.722216,0.526316
rescue,9.506119e-08,0.377843,8.295879,0.398191,0.631186,9.730797e-08,0.380953,8.649308,0.243902



=== Novel / ADC Stage 2 / Final ===
Stage2 pass: 25
Stage2 rescue: 5
Final top10: 10
Final top3: 3


,clone_id,novel_bucket,novel_final_score,pred_rescue_score,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_1619,pass,7.812331,0.358295,2.630590e-06,0.236190,3.356785,0.731156,2.544839e-06,0.164530,5.259631,1,0.854835,0.002945
1,CLONE_1039,pass,1.859526,0.245423,2.683238e-07,0.266199,2.900440,0.710503,2.564530e-07,0.324499,0.127177,0,0.511989,0.000000
2,CLONE_3581,pass,1.699847,0.251555,3.571934e-07,0.268148,3.505295,0.785921,3.901131e-07,0.252349,5.063234,1,0.419366,0.000000
3,CLONE_3822,pass,1.609323,0.301876,2.854552e-07,0.287347,3.031813,0.626934,3.452741e-07,0.156283,5.447983,1,0.229847,0.001302
4,CLONE_0894,pass,1.243008,0.253507,2.191406e-07,0.271822,3.658406,0.722470,2.730768e-07,0.084488,0.586578,1,0.270170,0.002871
5,CLONE_2358,pass,0.482645,0.418395,4.426706e-07,0.331023,4.866978,0.595937,5.671183e-07,0.204971,4.884648,1,0.254368,0.000000
6,CLONE_3969,pass,0.473056,0.376402,3.888828e-07,0.295752,5.623775,0.701205,4.846742e-07,0.168596,4.382455,1,0.169709,0.000000
7,CLONE_2660,pass,0.256155,0.334748,1.653555e-07,0.306763,4.497340,0.671642,2.022604e-07,0.208948,7.822708,1,0.019183,0.000000
8,CLONE_3922,pass,0.186742,0.318511,2.043583e-07,0.287936,5.320205,0.700448,2.008266e-07,0.315928,5.713897,0,0.055485,0.000000
9,CLONE_3465,pass,0.124785,0.385412,1.190938e-07,0.328251,4.120648,0.613127,1.404498e-07,0.232604,3.894179,1,0.019641,0.000000


## Step 7 — Compare decision modes

This step makes it easier to see how the selected clone sets differ
between biosimilar mode and novel / ADC mode.

In [9]:
# ==================================================
# Rescue-aware decision validation
# ==================================================

decision_validation = pd.DataFrame([
    summarize_final_selection("biosimilar", bio_top10, bio_top3, "bio_bucket"),
    summarize_final_selection("novel/ADC", novel_top10, novel_top3, "novel_bucket"),
])

display(decision_validation)

bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

print("=== Mode separation ===")
print("Top10 overlap:", len(bio_ids & novel_ids))
print("Biosimilar-only:", sorted(list(bio_ids - novel_ids))[:10])
print("Novel-only:", sorted(list(novel_ids - bio_ids))[:10])

,mode,top10_n,top3_n,top10_rescue_count,top3_rescue_count,top10_true_stable_rate,top3_true_stable_rate,top10_mean_true_late_qp,top10_mean_true_qp_drop,top10_mean_true_late_agg,top10_mean_pred_rescue_score
0,biosimilar,10,3,1,0,0.6,1.000000,2.455753e-06,0.276999,9.380556,0.609162
1,novel/ADC,10,3,0,0,0.8,0.666667,5.405085e-07,0.211320,4.318249,0.324412


=== Mode separation ===
Top10 overlap: 2
Biosimilar-only: ['CLONE_0550', 'CLONE_1942', 'CLONE_2689', 'CLONE_2759', 'CLONE_3144', 'CLONE_3254', 'CLONE_4335', 'CLONE_4625']
Novel-only: ['CLONE_0894', 'CLONE_1039', 'CLONE_2358', 'CLONE_2660', 'CLONE_3465', 'CLONE_3822', 'CLONE_3922', 'CLONE_3969']


## Step 8 — Decision validation block

This section validates whether the decision engine behaves reasonably before we move on to generator realism updates.

We check five things:

1. **Mode separation**
   - Are biosimilar and novel/ADC selections meaningfully different?

2. **Rescue volume**
   - Are rescue clone counts reasonable, rather than excessively large?

3. **Final selection quality**
   - Are the final Top-10 / Top-3 sets acceptably stable?

4. **Utility alignment**
   - Does predicted selection still overlap with true-utility winners?

5. **Pre-generator baseline**
   - These values serve as the baseline before applying late-aggregation realism patches.

This is the main validation step for the decision engine itself.

In [10]:
# --------------------------------------------------
# Step 8 — Decision validation block
# --------------------------------------------------

validation_rows = []

# ---------------------------
# Selection overlap / mode separation
# ---------------------------
bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

top10_overlap_count = len(bio_ids & novel_ids)
top10_overlap_frac = top10_overlap_count / max(1, TOP_N)

# ---------------------------
# Rescue counts
# ---------------------------
bio_rescue_n_stage1 = len(bio_stage1_rescue) if "bio_stage1_rescue" in globals() else np.nan
bio_rescue_n_stage2 = len(bio_stage2_rescue) if "bio_stage2_rescue" in globals() else np.nan

novel_rescue_n_stage1 = len(novel_stage1_rescue) if "novel_stage1_rescue" in globals() else np.nan
novel_rescue_n_stage2 = len(novel_stage2_rescue) if "novel_stage2_rescue" in globals() else np.nan

# ---------------------------
# Final selection quality
# ---------------------------
bio_top10_stable_rate = float(bio_top10["true_stable_label"].mean())
bio_top3_stable_rate = float(bio_top3["true_stable_label"].mean()) if "bio_top3" in globals() else np.nan

novel_top10_stable_rate = float(novel_top10["true_stable_label"].mean())
novel_top3_stable_rate = float(novel_top3["true_stable_label"].mean()) if "novel_top3" in globals() else np.nan

# ---------------------------
# Utility overlap
# ---------------------------
bio_util_overlap = topk_overlap(df["true_util_bio"], df["pred_util_bio"], TOP_N)
novel_util_overlap = topk_overlap(df["true_util_novel"], df["pred_util_novel"], TOP_N)

# ---------------------------
# Validation table
# ---------------------------
validation_rows.append({
    "check": "mode_overlap_top10",
    "biosimilar": np.nan,
    "novel_adc": np.nan,
    "combined": top10_overlap_count,
    "notes": f"{top10_overlap_frac:.2f} of final top10 overlaps between modes",
})

validation_rows.append({
    "check": "stage1_rescue_n",
    "biosimilar": bio_rescue_n_stage1,
    "novel_adc": novel_rescue_n_stage1,
    "combined": np.nan,
    "notes": "Too high may indicate thresholds are too loose",
})

validation_rows.append({
    "check": "stage2_rescue_n",
    "biosimilar": bio_rescue_n_stage2,
    "novel_adc": novel_rescue_n_stage2,
    "combined": np.nan,
    "notes": "Final rescue count should remain limited",
})

validation_rows.append({
    "check": "final_top10_stable_rate",
    "biosimilar": bio_top10_stable_rate,
    "novel_adc": novel_top10_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better",
})

validation_rows.append({
    "check": "final_top3_stable_rate",
    "biosimilar": bio_top3_stable_rate,
    "novel_adc": novel_top3_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better; should usually be >= top10 stable rate",
})

validation_rows.append({
    "check": "top10_true_utility_overlap",
    "biosimilar": bio_util_overlap,
    "novel_adc": novel_util_overlap,
    "combined": np.nan,
    "notes": "Ranking alignment with true utility",
})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,check,biosimilar,novel_adc,combined,notes
0,mode_overlap_top10,NaN,NaN,2.0,0.20 of final top10 overlaps between modes
1,stage1_rescue_n,160.0,123.000000,NaN,Too high may indicate thresholds are too loose
2,stage2_rescue_n,5.0,5.000000,NaN,Final rescue count should remain limited
3,final_top10_stable_rate,0.6,0.800000,NaN,Higher is better
4,final_top3_stable_rate,1.0,0.666667,NaN,Higher is better; should usually be >= top10 s...
5,top10_true_utility_overlap,0.3,0.100000,NaN,Ranking alignment with true utility


In [11]:
print("=== Decision validation summary ===")
print(f"Final top10 overlap between biosimilar and novel/ADC: {top10_overlap_count}/{TOP_N} ({top10_overlap_frac:.2%})")

print("\nRescue counts")
print(f"- Biosimilar: Stage1={bio_rescue_n_stage1}, Stage2={bio_rescue_n_stage2}")
print(f"- Novel/ADC: Stage1={novel_rescue_n_stage1}, Stage2={novel_rescue_n_stage2}")

print("\nFinal stable rates")
print(f"- Biosimilar: top10={bio_top10_stable_rate:.3f}, top3={bio_top3_stable_rate:.3f}")
print(f"- Novel/ADC: top10={novel_top10_stable_rate:.3f}, top3={novel_top3_stable_rate:.3f}")

print("\nUtility overlap")
print(f"- Biosimilar top10 utility overlap: {bio_util_overlap:.3f}")
print(f"- Novel/ADC top10 utility overlap: {novel_util_overlap:.3f}")

=== Decision validation summary ===
Final top10 overlap between biosimilar and novel/ADC: 2/10 (20.00%)

Rescue counts
- Biosimilar: Stage1=160, Stage2=5
- Novel/ADC: Stage1=123, Stage2=5

Final stable rates
- Biosimilar: top10=0.600, top3=1.000
- Novel/ADC: top10=0.800, top3=0.667

Utility overlap
- Biosimilar top10 utility overlap: 0.300
- Novel/ADC top10 utility overlap: 0.100


## Step 9 — Utility-based weight sweep (optional analysis)

This section is not the main deployment rule.

Instead, it helps us explore whether different utility weight combinations
could improve top-k clone selection.

This is useful for:
- comparing biosimilar vs novel priorities
- tuning future SDL decision rules
- stress-testing the decision engine

In [12]:
grid_bio = {
    "w_qp": [0.5, 1.0, 1.5, 2.0],
    "w_drop": [0.5, 1.0, 1.5, 2.0],
    "w_agg": [0.0, 0.1, 0.2, 0.3],
}

grid_novel = {
    "w_qp": [0.5, 1.0, 1.5],
    "w_drop": [0.5, 1.0, 1.5],
    "w_agg": [0.5, 1.0, 1.5, 2.0],
}

ELITE_FRACS = [0.10]
K_LIST = [5, 10, 20]

def mark_elite(df, util_col, frac):
    thr = df[util_col].quantile(1 - frac)
    return (df[util_col] >= thr).astype(int)

def run_sweep(base_df, mode_name, grid, true_util_col):
    rows = []
    work = base_df.copy()

    for w_qp in grid["w_qp"]:
        for w_drop in grid["w_drop"]:
            for w_agg in grid["w_agg"]:
                work["pred_util_tmp"] = (
                    w_qp * z(work["pred_late_qp"])
                    - w_drop * z(work["pred_qp_drop"])
                    - w_agg * z(work["pred_late_agg"])
                )

                for frac in ELITE_FRACS:
                    elite_col = f"elite_{int(frac*100)}"
                    work[elite_col] = mark_elite(work, true_util_col, frac)

                    for k in K_LIST:
                        rows.append({
                            "mode": mode_name,
                            "w_qp": w_qp,
                            "w_drop": w_drop,
                            "w_agg": w_agg,
                            "elite_frac": frac,
                            "k": k,
                            "precision@k": precision_at_k_df(work, "pred_util_tmp", elite_col, k),
                            "ndcg@k": ndcg_at_k_df(work, "pred_util_tmp", elite_col, k),
                        })
    return pd.DataFrame(rows)

bio_sweep = run_sweep(df, "biosimilar", grid_bio, "true_util_bio")
novel_sweep = run_sweep(df, "novel/ADC", grid_novel, "true_util_novel")

display(bio_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))
display(novel_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))

,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
3,biosimilar,0.5,0.5,0.1,0.1,5,1.0,1.0
6,biosimilar,0.5,0.5,0.2,0.1,5,1.0,1.0
9,biosimilar,0.5,0.5,0.3,0.1,5,1.0,1.0
33,biosimilar,0.5,1.5,0.3,0.1,5,1.0,1.0
66,biosimilar,1.0,1.0,0.2,0.1,5,1.0,1.0
69,biosimilar,1.0,1.0,0.3,0.1,5,1.0,1.0
129,biosimilar,1.5,1.5,0.3,0.1,5,1.0,1.0
0,biosimilar,0.5,0.5,0.0,0.1,5,0.8,1.0
12,biosimilar,0.5,1.0,0.0,0.1,5,0.8,1.0
15,biosimilar,0.5,1.0,0.1,0.1,5,0.8,1.0


,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
6,novel/ADC,0.5,0.5,1.5,0.1,5,0.8,0.904717
18,novel/ADC,0.5,1.0,1.5,0.1,5,0.8,0.904717
21,novel/ADC,0.5,1.0,2.0,0.1,5,0.8,0.904717
33,novel/ADC,0.5,1.5,2.0,0.1,5,0.8,0.904717
69,novel/ADC,1.0,1.5,2.0,0.1,5,0.8,0.760640
27,novel/ADC,0.5,1.5,1.0,0.1,5,0.6,0.906025
30,novel/ADC,0.5,1.5,1.5,0.1,5,0.6,0.906025
81,novel/ADC,1.5,0.5,2.0,0.1,5,0.6,0.906025
9,novel/ADC,0.5,0.5,2.0,0.1,5,0.6,0.885460
78,novel/ADC,1.5,0.5,1.5,0.1,5,0.6,0.885460


## Output of Notebook 04

This notebook converts predicted late-stage outcomes into rescue-aware clone selection decisions.

### Main outputs

- Biosimilar-mode pass / rescue / fail buckets
- Novel / ADC-mode pass / rescue / fail buckets
- Stage 2 re-ranked candidate pools
- Final Top-10 and Top-3 selections
- Rescue-aware decision validation
- Optional utility weight sweep

### Decision logic

- **Pass**
  - predicted to satisfy productivity, stability, and quality requirements
- **Rescue**
  - not an obvious pass clone
  - has strong predicted upside
  - has process-optimization potential
- **Fail**
  - insufficient predicted performance or excessive risk

### Pipeline meaning

- **Notebook 03 = prediction engine**
- **Notebook 04 = rescue-aware decision engine**

### Final baseline decision rule

The final decision engine uses pure ranking for Top-3 production candidates, while applying a minimal rescue guardrail only to the Top-10 biosimilar candidate pool.

This preserves production-candidate integrity while preventing high-upside rescue candidates from being completely excluded.

Current baseline settings:

- Biosimilar mode:
  - Top-10 uses ranking plus a minimal rescue guardrail
  - At least one rescue candidate is retained if no rescue clone appears naturally
  - Top-3 remains pure ranking

- Novel / ADC mode:
  - Top-10 remains pure ranking
  - No forced rescue inclusion
  - Top-3 remains pure ranking

This notebook is the bridge from ML prediction to practical CLD decision-making.